## Day 2 — Silver Layer (original-guide style + DQ rules table)

This notebook keeps the same overall flow as your code guide and adds a **Data Quality Rules Catalogue table**. The rule IDs, descriptions, and actions are defined once and then referenced in the validation logic.

**Before running:** confirm the Bronze schema and reference-table names in your Catalog Explorer. Your known Silver schema is `silver_schema`. This notebook does not invent admission/discharge timestamps or assume that operational statuses are hospital stay events.

## 1. Imports and table configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

CATALOG = "capstone_8hospital"

# CHANGE only if Catalog Explorer shows different schemas/table names.
BRONZE_SCHEMA = "hospital_schema"
SILVER_SCHEMA = "silver_schema"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

BRONZE_ENCOUNTER_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_encounter_events"
BRONZE_CDC_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_status_cdc"
BRONZE_SECONDARY_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_secondary_events"

# Confirm these two names and schemas against your actual reference tables.
BRONZE_DEPARTMENT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.department_bronze"
BRONZE_PATIENT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.patient_bronze"

SILVER_ENCOUNTER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_encounter_events"
SILVER_REJECTED_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_rejected_records"
SILVER_DQ_RULE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_dq_rule_catalogue"
SILVER_STATUS_HISTORY_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_status_history"
SILVER_CURRENT_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_encounter_current"
SILVER_SECONDARY_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_secondary_events"



## 2. Create the Data Quality Rules Catalogue table

The catalogue makes the rules visible in Databricks. No bill-amount or wait-time threshold is configured in this notebook.

In [0]:
rule_rows = [
    ("DQ-01", "encounter_id", "Required identifier is present", "REJECT", "Missing or blank encounter_id"),
    ("DQ-02", "event_timestamp", "Timestamp is valid", "REJECT", "Missing or unparseable event_timestamp"),
    ("DQ-03", "bill_amount", "Bill amount is numeric and greater than zero", "REJECT", "Invalid bill_amount"),
    ("DQ-04", "wait_minutes", "Wait minutes is numeric and non-negative", "REJECT", "Invalid wait_minutes"),
    ("DQ-05", "department_id", "Department key exists in reference table", "REJECT", "Unmatched department_id"),
    ("DQ-06", "patient_id", "Patient key exists in reference table", "REJECT", "Unmatched patient_id"),
    ("DQ-07", "encounter_id", "Duplicate encounter key after quality checks", "QUARANTINE_EXTRA","Duplicate encounter_id; one row retained"),
    ("DQ-08", "optional categorical fields", "Missing optional category is retained and flagged", "FLAG", "Missing optional categorical value"),
]

rule_schema = T.StructType([
    T.StructField("rule_id", T.StringType(), False),
    T.StructField("field_name", T.StringType(), False),
    T.StructField("rule_description", T.StringType(), False),
    T.StructField("action", T.StringType(), False),
    T.StructField("failure_reason", T.StringType(), False),
])

RULE_IDS = {row[0]: row[0] for row in rule_rows}

dq_rules = spark.createDataFrame(rule_rows, schema=rule_schema)

dq_rules = dq_rules.withColumn("updated_at", F.current_timestamp())

display(dq_rules.orderBy("rule_id"))

(dq_rules.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_DQ_RULE_TABLE))

print("Saved rule catalogue:", SILVER_DQ_RULE_TABLE)

rule_id,field_name,rule_description,action,failure_reason,updated_at
DQ-01,encounter_id,Required identifier is present,REJECT,Missing or blank encounter_id,2026-11-03T19:03:17.330Z
DQ-02,event_timestamp,Timestamp is valid,REJECT,Missing or unparseable event_timestamp,2026-11-03T19:03:17.330Z
DQ-03,bill_amount,Bill amount is numeric and greater than zero,REJECT,Invalid bill_amount,2026-11-03T19:03:17.330Z
DQ-04,wait_minutes,Wait minutes is numeric and non-negative,REJECT,Invalid wait_minutes,2026-11-03T19:03:17.330Z
DQ-05,department_id,Department key exists in reference table,REJECT,Unmatched department_id,2026-11-03T19:03:17.330Z
DQ-06,patient_id,Patient key exists in reference table,REJECT,Unmatched patient_id,2026-11-03T19:03:17.330Z
DQ-07,encounter_id,Duplicate encounter key after quality checks,QUARANTINE_EXTRA,Duplicate encounter_id; one row retained,2026-11-03T19:03:17.330Z
DQ-08,optional categorical fields,Missing optional category is retained and flagged,FLAG,Missing optional categorical value,2026-11-03T19:03:17.330Z


Saved rule catalogue: capstone_8hospital.silver_schema.silver_dq_rule_catalogue


## 3. Read Bronze tables and inspect schemas

In [0]:
encounters = spark.table(BRONZE_ENCOUNTER_TABLE)
cdc = spark.table(BRONZE_CDC_TABLE)
secondary = spark.table(BRONZE_SECONDARY_TABLE)
departments = spark.table(BRONZE_DEPARTMENT_TABLE)
patients = spark.table(BRONZE_PATIENT_TABLE)

encounters.printSchema()
cdc.printSchema()
secondary.printSchema()
departments.printSchema()
patients.printSchema()

display(spark.createDataFrame(
    [
        ("Bronze encounter events", encounters.count()),
        ("Bronze status CDC", cdc.count()),
        ("Bronze secondary events", secondary.count()),
        ("Department reference", departments.count()),
        ("Patient reference", patients.count()),
    ],
    ["table_name", "row_count"]
))

root
 |-- encounter_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)
 |-- department_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- visit_type: string (nullable = true)
 |-- encounter_status: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- bill_amount: double (nullable = true)
 |-- wait_minutes: integer (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_schema_version: string (nullable = true)
 |-- _rescued_data: string (nullable = true)

root
 |-- encounter_id: string (nullable = true)
 |-- change_timestamp: timestamp (nullable = true)
 |-- previous_status: string (nullable = true)
 |-- new_status: string (nullable = true)
 |-- change_reason: string (nullable = true)
 |-- source_sequence: integer (nullable = true)
 |-- source_system: string (nullable = true)

root
 |-- event_id: string (null

table_name,row_count
Bronze encounter events,11540
Bronze status CDC,520
Bronze secondary events,310
Department reference,110
Patient reference,150


## 4. Cast and standardize encounter data

In [0]:
typed = (
    encounters
    .withColumn("encounter_id", F.trim(F.col("encounter_id").cast("string")))
    .withColumn("patient_id", F.trim(F.col("patient_id").cast("string")))
    .withColumn("department_id", F.trim(F.col("department_id").cast("string")))
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
    .withColumn("ingest_timestamp", F.to_timestamp("ingest_timestamp"))
    .withColumn("bill_amount", F.col("bill_amount").cast("double"))
    .withColumn("wait_minutes", F.col("wait_minutes").cast("int"))
)

print("Typed rows:", typed.count())
display(typed.limit(10))

Typed rows: 11540


encounter_id,event_timestamp,ingest_timestamp,department_id,patient_id,visit_type,encounter_status,priority,bill_amount,wait_minutes,doctor_id,source_system,source_schema_version,_rescued_data
ENCOUNTER-0011352,2026-08-01T12:46:00.000Z,2026-08-01T12:59:00.000Z,DEPARTMENT_ID-0022,PATIENT_ID-0102,INPATIENT,COMPLETED,USD,1299.92,60,DOCTOR_ID-0042,ADLS_EVENT,v2,null
ENCOUNTER-0011375,2026-08-01T14:17:00.000Z,2026-08-01T14:33:00.000Z,DEPARTMENT_ID-0045,PATIENT_ID-0125,OUTPATIENT,FAILED,USD,1162.34,14,DOCTOR_ID-0065,ADLS_EVENT,v2,null
ENCOUNTER-0007053,2026-07-20T14:08:00.000Z,2026-07-20T14:33:00.000Z,DEPARTMENT_ID-0013,PATIENT_ID-0003,OUTPATIENT,COMPLETED,USD,72.97,123,DOCTOR_ID-0033,ADLS_EVENT,v2,null
ENCOUNTER-0008647,2026-07-25T00:25:00.000Z,2026-07-25T00:46:00.000Z,DEPARTMENT_ID-0067,PATIENT_ID-0097,INPATIENT,OPEN,USD,961.5,151,DOCTOR_ID-0002,ADLS_EVENT,v2,null
ENCOUNTER-0005083,2026-07-15T02:48:00.000Z,2026-07-15T03:13:00.000Z,DEPARTMENT_ID-0023,PATIENT_ID-0133,INPATIENT,COMPLETED,USD,513.87,149,DOCTOR_ID-0013,ADLS_EVENT,v2,null
ENCOUNTER-0000073,2026-07-01T04:49:00.000Z,2026-07-01T05:02:00.000Z,DEPARTMENT_ID-0073,PATIENT_ID-0073,OUTPATIENT,FAILED,USD,1262.76,44,DOCTOR_ID-0008,ADLS_EVENT,v2,null
ENCOUNTER-0003374,2026-07-10T08:53:00.000Z,2026-07-10T09:05:00.000Z,DEPARTMENT_ID-0074,PATIENT_ID-0074,OUTPATIENT,FAILED,USD,586.49,51,DOCTOR_ID-0059,ADLS_EVENT,v2,null
ENCOUNTER-0005078,2026-07-15T02:30:00.000Z,2026-07-15T02:49:00.000Z,DEPARTMENT_ID-0018,PATIENT_ID-0128,OUTPATIENT,PENDING,USD,177.55,32,DOCTOR_ID-0008,ADLS_EVENT,v2,null
ENCOUNTER-0004618,2026-07-13T19:50:00.000Z,2026-07-13T20:04:00.000Z,DEPARTMENT_ID-0108,PATIENT_ID-0118,OUTPATIENT,OPEN,USD,1373.1,2,DOCTOR_ID-0003,ADLS_EVENT,v2,null
ENCOUNTER-0003298,2026-07-10T03:50:00.000Z,2026-07-10T04:10:00.000Z,DEPARTMENT_ID-0108,PATIENT_ID-0148,OUTPATIENT,FAILED,USD,1153.18,146,DOCTOR_ID-0048,ADLS_EVENT,v2,null


## 5. Apply DQ-01 to DQ-06 using the rule IDs from the catalogue

In [0]:
# Rule IDs are defined in silver_dq_rule_catalogue above and reused in the checks.
# Preserve the first applicable rejection reason for each row.

checked = (
    typed
    .withColumn(
        "rule_id",
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), F.lit(RULE_IDS["DQ-01"]))
         .when(F.col("event_timestamp").isNull(), F.lit(RULE_IDS["DQ-02"]))
         .when(F.col("bill_amount").isNull() | (F.col("bill_amount") <= 0), F.lit(RULE_IDS["DQ-03"]))
         .when(F.col("wait_minutes").isNull() | (F.col("wait_minutes") < 0), F.lit(RULE_IDS["DQ-04"]))
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rule_id") == "DQ-01", F.lit("Missing or blank encounter_id"))
         .when(F.col("rule_id") == "DQ-02", F.lit("Missing or unparseable event_timestamp"))
         .when(F.col("rule_id") == "DQ-03", F.lit("bill_amount must be numeric and greater than zero"))
         .when(F.col("rule_id") == "DQ-04", F.lit("wait_minutes must be numeric and non-negative"))
    )
)

department_keys = departments.select(F.trim(F.col("department_id").cast("string")).alias("department_id")).dropDuplicates()
patient_keys = patients.select(F.trim(F.col("patient_id").cast("string")).alias("patient_id")).dropDuplicates()

with_refs = (
    checked
    .join(department_keys.withColumn("_department_found", F.lit(True)), "department_id", "left")
    .join(patient_keys.withColumn("_patient_found", F.lit(True)), "patient_id", "left")
    .withColumn(
        "rule_id",
        F.when(F.col("rule_id").isNotNull(), F.col("rule_id"))
         .when(F.col("_department_found").isNull(), F.lit(RULE_IDS["DQ-05"]))
         .when(F.col("_patient_found").isNull(), F.lit(RULE_IDS["DQ-06"]))
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rejection_reason").isNotNull(), F.col("rejection_reason"))
         .when(F.col("rule_id") == "DQ-05", F.lit("department_id not found in department reference"))
         .when(F.col("rule_id") == "DQ-06", F.lit("patient_id not found in patient reference"))
    )
    .withColumn("rejected_at", F.current_timestamp())
)

rejected_basic = with_refs.filter(F.col("rule_id").isNotNull())
valid_basic = with_refs.filter(F.col("rule_id").isNull())

print("Rows before deduplication:", valid_basic.count())
print("Rejected by DQ-01 to DQ-06:", rejected_basic.count())

Rows before deduplication: 11540
Rejected by DQ-01 to DQ-06: 0


## 6. Apply DQ-07 deduplication and DQ-08 missing-category flag

This follows the guide's business-key assumption: keep one row per `encounter_id`, choosing the most recently ingested row. Confirm that the official task defines duplicates by encounter key rather than exact-row equality.

In [0]:
order_columns = [F.col("ingest_timestamp").desc_nulls_last()]
if "_source_file" in valid_basic.columns:
    order_columns.append(F.col("_source_file").asc_nulls_last())

dedup_window = Window.partitionBy("encounter_id").orderBy(*order_columns)
ranked = valid_basic.withColumn("_row_num", F.row_number().over(dedup_window))

duplicate_rows = (
    ranked.filter(F.col("_row_num") > 1)
    .withColumn("rule_id", F.lit(RULE_IDS["DQ-07"]))
    .withColumn("rejection_reason", F.lit("Duplicate encounter_id; most recently ingested row retained"))
    .withColumn("rejected_at", F.current_timestamp())
)

deduplicated = ranked.filter(F.col("_row_num") == 1).drop(
    "_row_num", "_department_found", "_patient_found"
)

rejected_final = rejected_basic.unionByName(duplicate_rows, allowMissingColumns=True)

# DQ-08: identify missing optional categorical values without rejecting them.
optional_candidates = [c for c in ["visit_type", "encounter_status", "priority", "source_system"] if c in deduplicated.columns]
missing_optional_expr = None
for c in optional_candidates:
    expr = F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")
    missing_optional_expr = expr if missing_optional_expr is None else (missing_optional_expr | expr)

if missing_optional_expr is None:
    missing_optional_expr = F.lit(False)

silver_encounters = (
    deduplicated
    .withColumn("missing_optional_category_flag", missing_optional_expr)
    .withColumn(
        "dq_status",
        F.when(F.col("missing_optional_category_flag"), F.lit("FLAGGED"))
         .otherwise(F.lit("VALID"))
    )
)

print("Silver encounter rows after deduplication:", silver_encounters.count())
print("Rejected rows including duplicates:", rejected_final.count())
display(silver_encounters.groupBy("dq_status", "missing_optional_category_flag").count())

Silver encounter rows after deduplication: 11500
Rejected rows including duplicates: 40


dq_status,missing_optional_category_flag,count
VALID,false,11488
FLAGGED,true,12


## 7. Write Silver encounter and rejected-record tables

In [0]:
(silver_encounters.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_ENCOUNTER_TABLE))

(rejected_final.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_REJECTED_TABLE))

print("Saved:", SILVER_ENCOUNTER_TABLE)
print("Saved:", SILVER_REJECTED_TABLE)

Saved: capstone_8hospital.silver_schema.silver_encounter_events
Saved: capstone_8hospital.silver_schema.silver_rejected_records


## 8. CDC: inspect real columns before processing

Your guide's CDC code assumes `event_timestamp`, but the source described in the project uses `change_timestamp`. This cell checks the actual schema and uses `change_timestamp` when present. It does not treat operational workflow statuses as admission/transfer/discharge events.

In [0]:
cdc.printSchema()
print("CDC columns:", cdc.columns)
display(cdc.limit(10))

# Select the real timestamp column without renaming the source data blindly.
if "change_timestamp" in cdc.columns:
    cdc_time_col = "change_timestamp"
elif "event_timestamp" in cdc.columns:
    cdc_time_col = "event_timestamp"
else:
    raise ValueError("CDC table has neither change_timestamp nor event_timestamp. Check the source schema.")

required_cdc_cols = ["encounter_id", "new_status", "source_sequence", cdc_time_col]
missing_cdc_cols = [c for c in required_cdc_cols if c not in cdc.columns]
if missing_cdc_cols:
    raise ValueError(f"CDC source is missing required columns: {missing_cdc_cols}")

cdc_typed = (
    cdc
    .withColumn("encounter_id", F.trim(F.col("encounter_id").cast("string")))
    .withColumn("source_sequence", F.col("source_sequence").cast("long"))
    .withColumn("status_changed_at", F.to_timestamp(F.col(cdc_time_col)))
)

cdc_checked = (
    cdc_typed
    .withColumn(
        "rule_id",
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), F.lit("CDC-01"))
         .when(F.col("new_status").isNull(), F.lit("CDC-02"))
         .when(F.col("source_sequence").isNull(), F.lit("CDC-03"))
         .when(F.col("status_changed_at").isNull(), F.lit("CDC-04"))
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rule_id") == "CDC-01", F.lit("Missing encounter_id"))
         .when(F.col("rule_id") == "CDC-02", F.lit("Missing new_status"))
         .when(F.col("rule_id") == "CDC-03", F.lit("Missing or invalid source_sequence"))
         .when(F.col("rule_id") == "CDC-04", F.lit("Missing or invalid status timestamp"))
    )
    .withColumn("rejected_at", F.current_timestamp())
)

cdc_rejected = cdc_checked.filter(F.col("rule_id").isNotNull())
cdc_valid = cdc_checked.filter(F.col("rule_id").isNull())
print("Valid CDC rows:", cdc_valid.count(), "Rejected CDC rows:", cdc_rejected.count())

root
 |-- encounter_id: string (nullable = true)
 |-- change_timestamp: timestamp (nullable = true)
 |-- previous_status: string (nullable = true)
 |-- new_status: string (nullable = true)
 |-- change_reason: string (nullable = true)
 |-- source_sequence: integer (nullable = true)
 |-- source_system: string (nullable = true)

CDC columns: ['encounter_id', 'change_timestamp', 'previous_status', 'new_status', 'change_reason', 'source_sequence', 'source_system']


encounter_id,change_timestamp,previous_status,new_status,change_reason,source_sequence,source_system
ENCOUNTER-0000001,2026-08-10T00:00:00.000Z,PENDING,CANCELLED,OPERATIONS_UPDATE,1,ADLS_CDC
ENCOUNTER-0000018,2026-08-10T00:01:00.000Z,OPEN,COMPLETED,CUSTOMER_ACTION,2,ADLS_CDC
ENCOUNTER-0000035,2026-08-10T00:02:00.000Z,COMPLETED,REVERSED,SYSTEM_REVIEW,3,ADLS_CDC
ENCOUNTER-0000052,2026-08-10T00:03:00.000Z,PENDING,COMPLETED,OPERATIONS_UPDATE,4,ADLS_CDC
ENCOUNTER-0000069,2026-08-10T00:04:00.000Z,COMPLETED,COMPLETED,SYSTEM_REVIEW,5,ADLS_CDC
ENCOUNTER-0000086,2026-08-10T00:05:00.000Z,FAILED,COMPLETED,CUSTOMER_ACTION,6,ADLS_CDC
ENCOUNTER-0000103,2026-08-10T00:06:00.000Z,COMPLETED,REVERSED,SYSTEM_REVIEW,7,ADLS_CDC
ENCOUNTER-0000120,2026-08-10T00:07:00.000Z,FAILED,CANCELLED,SYSTEM_REVIEW,8,ADLS_CDC
ENCOUNTER-0000137,2026-08-10T00:08:00.000Z,OPEN,CANCELLED,SYSTEM_REVIEW,9,ADLS_CDC
ENCOUNTER-0000154,2026-08-10T00:09:00.000Z,COMPLETED,CANCELLED,OPERATIONS_UPDATE,10,ADLS_CDC


Valid CDC rows: 520 Rejected CDC rows: 0


## 9. Build status history and latest status

The sequence is used first to select the latest state. Duplicate `(encounter_id, source_sequence)` pairs are displayed for inspection before one row is retained.

In [0]:
cdc_history = cdc_valid.dropDuplicates(["encounter_id", "source_sequence"])

(cdc_history.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_STATUS_HISTORY_TABLE))

status_window = Window.partitionBy("encounter_id").orderBy(
    F.col("source_sequence").desc(),
    F.col("status_changed_at").desc()
)

latest_status = (
    cdc_history
    .withColumn("_rn", F.row_number().over(status_window))
    .filter(F.col("_rn") == 1)
    .select(
        "encounter_id",
        F.col("new_status").alias("current_status"),
        F.col("status_changed_at")
    )
)

base_encounters = spark.table(SILVER_ENCOUNTER_TABLE)
for col_name in ["current_status", "status_changed_at"]:
    if col_name in base_encounters.columns:
        base_encounters = base_encounters.drop(col_name)

current_encounters = base_encounters.join(latest_status, "encounter_id", "left")

(current_encounters.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_CURRENT_TABLE))

print("Status history rows:", spark.table(SILVER_STATUS_HISTORY_TABLE).count())
print("Current encounter rows:", spark.table(SILVER_CURRENT_TABLE).count())
display(spark.table(SILVER_CURRENT_TABLE).groupBy("current_status").count().orderBy(F.col("count").desc()))



Status history rows: 520
Current encounter rows: 11500


current_status,count
null,10980
CANCELLED,185
REVERSED,172
COMPLETED,163


## 10. Secondary operational events

The exact secondary-event schema and rule requirements must be checked first. This conservative starter validates that the event has an encounter key and that the key exists in Silver encounters. Add other required checks from the official task before treating this table as complete.

In [0]:
secondary.printSchema()
display(secondary.limit(10))

silver_encounter_keys = spark.table(SILVER_ENCOUNTER_TABLE).select("encounter_id").dropDuplicates()

if "encounter_id" not in secondary.columns:
    raise ValueError("Secondary-event table does not contain encounter_id. Update the key after checking printSchema().")

secondary_checked = (
    secondary
    .withColumn("encounter_id", F.trim(F.col("encounter_id").cast("string")))
    .join(silver_encounter_keys.withColumn("_encounter_found", F.lit(True)), "encounter_id", "left")
    .withColumn(
        "rule_id",
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), F.lit("SEC-01"))
         .when(F.col("_encounter_found").isNull(), F.lit("SEC-02"))
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rule_id") == "SEC-01", F.lit("Missing encounter_id"))
         .when(F.col("rule_id") == "SEC-02", F.lit("Encounter key not found in Silver encounters"))
    )
    .withColumn("rejected_at", F.current_timestamp())
)

secondary_rejected = secondary_checked.filter(F.col("rule_id").isNotNull())
silver_secondary = secondary_checked.filter(F.col("rule_id").isNull()).drop("_encounter_found")

(silver_secondary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_SECONDARY_TABLE))

print("Secondary Bronze rows:", secondary.count())
print("Secondary Silver rows:", spark.table(SILVER_SECONDARY_TABLE).count())
print("Secondary rejected rows (not saved as a table):", secondary_rejected.count())


root
 |-- event_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- related_entity_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_value: double (nullable = true)
 |-- source_system: string (nullable = true)



event_id,encounter_id,event_timestamp,related_entity_id,event_type,event_value,source_system
SEC-000001,ENCOUNTER-0000001,2026-07-01T00:01:00.000Z,DEPARTMENT_ID-0001,ALERT,65.11,ADLS_AUX
SEC-000002,ENCOUNTER-0000032,2026-07-01T02:06:00.000Z,DEPARTMENT_ID-0032,ALERT,48.39,ADLS_AUX
SEC-000003,ENCOUNTER-0000063,2026-07-01T04:08:00.000Z,DEPARTMENT_ID-0063,CHECK,50.19,ADLS_AUX
SEC-000004,ENCOUNTER-0000094,2026-07-01T06:14:00.000Z,DEPARTMENT_ID-0094,CHECK,96.95,ADLS_AUX
SEC-000005,ENCOUNTER-0000125,2026-07-01T08:16:00.000Z,DEPARTMENT_ID-0015,ALERT,49.92,ADLS_AUX
SEC-000006,ENCOUNTER-0000156,2026-07-01T10:20:00.000Z,DEPARTMENT_ID-0046,CHECK,69.62,ADLS_AUX
SEC-000007,ENCOUNTER-0000187,2026-07-01T12:24:00.000Z,DEPARTMENT_ID-0077,REVIEW,95.45,ADLS_AUX
SEC-000008,ENCOUNTER-0000218,2026-07-01T14:29:00.000Z,DEPARTMENT_ID-0108,REVIEW,92.51,ADLS_AUX
SEC-000009,ENCOUNTER-0000249,2026-07-01T16:33:00.000Z,DEPARTMENT_ID-0029,ALERT,23.69,ADLS_AUX
SEC-000010,ENCOUNTER-0000280,2026-07-01T18:37:00.000Z,DEPARTMENT_ID-0060,ALERT,23.53,ADLS_AUX


Secondary Bronze rows: 310
Secondary Silver rows: 310
Secondary rejected rows (not saved as a table): 0


## 11. Final row-count summary

In [0]:
# Compact final dashboard: counts only.
# Detailed rejected rows, duplicate samples, flags, and status distributions
# are displayed in the sections where those checks are performed.

validation_rows = [
    ("Bronze encounter events", spark.table(BRONZE_ENCOUNTER_TABLE).count()),
    ("Silver encounter events", spark.table(SILVER_ENCOUNTER_TABLE).count()),
    ("Silver rejected records", spark.table(SILVER_REJECTED_TABLE).count()),
    ("Bronze status CDC", spark.table(BRONZE_CDC_TABLE).count()),
    ("Silver status history", spark.table(SILVER_STATUS_HISTORY_TABLE).count()),
    ("Silver current encounters", spark.table(SILVER_CURRENT_TABLE).count()),
    ("Bronze secondary events", spark.table(BRONZE_SECONDARY_TABLE).count()),
    ("Silver secondary events", spark.table(SILVER_SECONDARY_TABLE).count()),
]

display(spark.createDataFrame(validation_rows, ["validation_check", "row_count"]))

validation_check,row_count
Bronze encounter events,11540
Silver encounter events,11500
Silver rejected records,40
Bronze status CDC,520
Silver status history,520
Silver current encounters,11500
Bronze secondary events,310
Silver secondary events,310
